In [110]:
#Necessary Libraries
import requests
from bs4 import BeautifulSoup
import html
import pandas as pd
import re

API REQUEST


In [111]:
url = "https://boards-api.greenhouse.io/v1/boards/tide/jobs?content=true"

response = requests.get(url)

print("Status code:", response.status_code)
if response.status_code!=200:
    print("could not access this greenhouse board")
    print(response.text[:500])

Status code: 200


GETTING JOBS THROUGH API

In [112]:
data = response.json()

jobs = data["jobs"]

print("Number of jobs:", len(jobs))

Number of jobs: 84


BUILDING DATA SET

In [113]:
job_data = []

for job in jobs:

    decoded_content = html.unescape(job["content"])

    soup = BeautifulSoup(decoded_content, "html.parser")

    clean_text = soup.get_text(separator=" ", strip=True)

    record = {
        "job_id": job["id"],
        "title": job["title"],
        "company": job["company_name"],
        "location": job["location"]["name"],
        "url": job["absolute_url"],
        "published": job["first_published"],
        "updated": job["updated_at"],
        "description": clean_text
    }

    job_data.append(record)

CONVERSION TO PANDAS DATAFRAME

In [114]:
df = pd.DataFrame(job_data)

df.head(10)


,job_id,title,company,location,url,published,updated,description
0,7681876003,Account Executive,Careers at Tide,United Kingdom,https://job-boards.greenhouse.io/tide/jobs/768...,2026-03-27T09:45:37-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
1,7514127003,Account Executive – SME Credit,Careers at Tide,"Berlin, Germany",https://job-boards.greenhouse.io/tide/jobs/751...,2025-10-31T10:40:09-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
2,7685198003,Asset Finance Specialist,Careers at Tide,United Kingdom,https://job-boards.greenhouse.io/tide/jobs/768...,2026-04-01T08:43:11-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
3,7769443003,Business Development Executive - French Speaking,Careers at Tide,"India, Hyderabad",https://job-boards.greenhouse.io/tide/jobs/776...,2026-06-12T05:01:36-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
4,7821236003,Business Development Executive - French Speaking,Careers at Tide,Bulgaria,https://job-boards.greenhouse.io/tide/jobs/782...,2026-07-30T04:09:40-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
5,7598880003,Business Development Specialist (International...,Careers at Tide,"India, Delhi NCR",https://job-boards.greenhouse.io/tide/jobs/759...,2026-01-21T06:59:06-05:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
6,7081017003,Business Development Specialist (Outbound- Int...,Careers at Tide,"India, Delhi NCR",https://job-boards.greenhouse.io/tide/jobs/708...,2025-09-08T06:25:07-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
7,7983019003,Business Operations Manager,Careers at Tide,"Berlin, Germany",https://job-boards.greenhouse.io/tide/jobs/798...,2026-09-01T02:44:08-04:00,2026-09-01T02:44:08-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
8,7822661003,Commercial Services - UK Country Commercial Lead,Careers at Tide,United Kingdom,https://job-boards.greenhouse.io/tide/jobs/782...,2026-07-31T10:07:17-04:00,2026-08-31T05:47:35-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
9,7766788003,Customer Support Executive with German (Onboar...,Careers at Tide,Bulgaria,https://job-boards.greenhouse.io/tide/jobs/776...,2026-06-11T08:00:17-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."


In [115]:
df["company"].unique().sum()

'Careers at Tide'

CREATING FUNCTION TO GET  DATA OF DIFFERENT COMPANIES


In [116]:
def get_greenhouse_jobs(board_token):

    url = f"https://boards-api.greenhouse.io/v1/boards/{board_token}/jobs?content=true"

    response = requests.get(url)

    print("Status code:", response.status_code)

    if response.status_code != 200:
        print("Could not access this Greenhouse board.")
        return None

    data = response.json()

    jobs = data["jobs"]

    print("Jobs found:", len(jobs))

    job_data = []

    for job in jobs:

        decoded_content = html.unescape(job.get("content", ""))

        soup = BeautifulSoup(decoded_content, "html.parser")

        clean_text = soup.get_text(separator=" ", strip=True)

        record = {
            "job_id": job.get("id"),
            "title": job.get("title"),
            "company": job.get("company_name"),
            "location": job.get("location", {}).get("name"),
            "url": job.get("absolute_url"),
            "published": job.get("first_published"),
            "updated": job.get("updated_at"),
            "description": clean_text
        }

        job_data.append(record)

    df = pd.DataFrame(job_data)

    return df

TESTING THE FUNCTION

In [117]:
tide_df=get_greenhouse_jobs("tide")

Status code: 200
Jobs found: 84


In [118]:
tide_df.shape

(84, 8)

In [119]:
tide_df.head()

,job_id,title,company,location,url,published,updated,description
0,7681876003,Account Executive,Careers at Tide,United Kingdom,https://job-boards.greenhouse.io/tide/jobs/768...,2026-03-27T09:45:37-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
1,7514127003,Account Executive – SME Credit,Careers at Tide,"Berlin, Germany",https://job-boards.greenhouse.io/tide/jobs/751...,2025-10-31T10:40:09-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
2,7685198003,Asset Finance Specialist,Careers at Tide,United Kingdom,https://job-boards.greenhouse.io/tide/jobs/768...,2026-04-01T08:43:11-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
3,7769443003,Business Development Executive - French Speaking,Careers at Tide,"India, Hyderabad",https://job-boards.greenhouse.io/tide/jobs/776...,2026-06-12T05:01:36-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."
4,7821236003,Business Development Executive - French Speaking,Careers at Tide,Bulgaria,https://job-boards.greenhouse.io/tide/jobs/782...,2026-07-30T04:09:40-04:00,2026-08-28T06:01:23-04:00,"A BOUT TIDE At Tide, we help SMEs save time an..."


ADDING SECOND COMPANY

In [120]:
gitlab_df = get_greenhouse_jobs("gitlab")

Status code: 200
Jobs found: 231


In [121]:
gitlab_df.head(),gitlab_df.shape

(       job_id                                      title company  \
 0  8503792002                  Account Executive - Italy  GitLab   
 1  8556658002                                AI Engineer  GitLab   
 2  8638232002               AI Transformation Owner, CRO  GitLab   
 3  8716179002  AI Transformation Owner, Product & Design  GitLab   
 4  8631068002  Area Vice President - Financial Services   GitLab   
 
                                             location  \
 0                                      Remote, Italy   
 1                                  Remote, Bangalore   
 2                              Remote, United States   
 3  Remote, Canada; Remote, United Kingdom; Remote...   
 4                                         Remote, US   
 
                                                  url  \
 0  https://job-boards.greenhouse.io/gitlab/jobs/8...   
 1  https://job-boards.greenhouse.io/gitlab/jobs/8...   
 2  https://job-boards.greenhouse.io/gitlab/jobs/8...   
 3  https://

GENERATING MULTI COMPANY DATASET

In [122]:
all_jobs=pd.concat(
    [tide_df,gitlab_df],
    ignore_index=True
)

In [123]:
all_jobs.shape

(315, 8)

CHECKING FOR DUPLICATES

In [124]:
all_jobs["job_id"].duplicated().sum()

np.int64(0)

checking for unique ids

In [125]:
all_jobs["job_id"].nunique()

315

In [126]:
all_jobs.shape

(315, 8)

In [127]:
all_jobs["company"].value_counts()

company
GitLab             231
Careers at Tide     84
Name: count, dtype: int64

ADDED MORE COMPANIES TO THE DATASET

In [128]:
companies = [
    "tide",
    "gitlab",
    "figma",
    "stripe",
    "twilio",
    "webflow",
    "customerio",
    "datadog",
    "bigid",
    "karbon"

]

In [129]:
company_dfs=[]
for company in companies:
    df=get_greenhouse_jobs(company)
    if df is not None:
        company_dfs.append(df)

Status code: 200
Jobs found: 84
Status code: 200
Jobs found: 231
Status code: 200
Jobs found: 157
Status code: 200
Jobs found: 612
Status code: 200
Jobs found: 144
Status code: 200
Jobs found: 29
Status code: 200
Jobs found: 28
Status code: 200
Jobs found: 444
Status code: 200
Jobs found: 7
Status code: 200
Jobs found: 15


In [130]:
all_jobs=pd.concat(company_dfs,ignore_index=True)
print("Total jobs:",len(all_jobs))
print(all_jobs["company"].value_counts())

Total jobs: 1751
company
Stripe             612
Datadog            444
GitLab             231
Figma              157
Twilio             144
Careers at Tide     84
Webflow             29
Customer.io         28
Karbon              15
BigID                7
Name: count, dtype: int64


DATA CLEANING

In [131]:
print("Rows",all_jobs.shape[0])
print("Columns",all_jobs.shape[1])
all_jobs.info()

Rows 1751
Columns 8
<class 'pandas.DataFrame'>
RangeIndex: 1751 entries, 0 to 1750
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   job_id       1751 non-null   int64
 1   title        1751 non-null   str  
 2   company      1751 non-null   str  
 3   location     1751 non-null   str  
 4   url          1751 non-null   str  
 5   published    1751 non-null   str  
 6   updated      1751 non-null   str  
 7   description  1751 non-null   str  
dtypes: int64(1), str(7)
memory usage: 109.6 KB


CHECKING NULL VALUES

In [132]:
all_jobs.isnull().sum()

job_id         0
title          0
company        0
location       0
url            0
published      0
updated        0
description    0
dtype: int64

CHECKING DUPLICATED VALUES

In [133]:
print("Duplicated job ids",all_jobs["job_id"].duplicated().sum())

Duplicated job ids 0


In [134]:
all_jobs["company"].value_counts()

company
Stripe             612
Datadog            444
GitLab             231
Figma              157
Twilio             144
Careers at Tide     84
Webflow             29
Customer.io         28
Karbon              15
BigID                7
Name: count, dtype: int64

In [135]:
all_jobs["location"].value_counts().head(20)

location
New York, New York, USA                                90
San Francisco, CA • New York, NY • United States       82
Remote - US                                            66
Tokyo, Japan                                           53
Singapore                                              48
Paris, France                                          39
Bangalore, India                                       39
Remote, United States                                  37
Remote, Canada; Remote, United States                  32
Dublin                                                 30
Bengaluru                                              30
London                                                 27
Americas Remote                                        24
United Kingdom                                         23
Boston, Massachusetts, USA; New York, New York, USA    23
Mexico City                                            21
Remote, United Kingdom                                 20
N/A  

In [136]:
all_jobs["title"].value_counts().head(30)

title
Strategic Account Executive                               21
Enterprise Sales Executive                                 9
Commercial Account Executive                               7
Software Engineer, New Grad                                6
Engineering Manager                                        5
Senior Staff Software Engineer, Agentic Platform           5
Staff Software Engineer, Agentic Platform                  5
Senior Software Engineer                                   5
Staff Software Engineer                                    5
Machine Learning Engineer                                  5
Technical Account Manager                                  5
Enterprise Customer Success Manager                        5
Mid Market Account Executive                               5
Director of Mobile and Web Platform                        4
Customer Success Manager                                   4
Senior Engineering Manager V&V Media                       4
Enterprise Sales E

CHECKING DISTRIBUTION OF DESCRIPTION LENGTH

In [137]:
all_jobs["description"].str.len().describe()

count     1751.000000
mean      5906.550543
std       1950.496444
min       2227.000000
25%       4400.000000
50%       5579.000000
75%       7400.500000
max      14791.000000
Name: description, dtype: float64

CHECKING  10 SHORTEST DESCRIPTIONS

In [138]:
all_jobs["description"].str.len().sort_values().head(10)

1744    2227
541     2239
936     2327
958     2340
901     2423
489     2479
833     2494
728     2552
980     2556
921     2557
Name: description, dtype: int64

CHECKING LONG DESCRIPTIONS

In [139]:
all_jobs["description"].str.len().sort_values(ascending=False).head(10)

209     14791
287     14088
135     13406
1207    12524
1100    12430
1041    12372
65      11695
169     11610
179     11608
87      11581
Name: description, dtype: int64

CONVERTING DATES FROM STRINGS TO DATETIME 

In [140]:
all_jobs["published"]=pd.to_datetime(all_jobs["published"],utc=True)
all_jobs["updated"]=pd.to_datetime(all_jobs["updated"],utc=True)

In [141]:
all_jobs.info()

<class 'pandas.DataFrame'>
RangeIndex: 1751 entries, 0 to 1750
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   job_id       1751 non-null   int64              
 1   title        1751 non-null   str                
 2   company      1751 non-null   str                
 3   location     1751 non-null   str                
 4   url          1751 non-null   str                
 5   published    1751 non-null   datetime64[us, UTC]
 6   updated      1751 non-null   datetime64[us, UTC]
 7   description  1751 non-null   str                
dtypes: datetime64[us, UTC](2), int64(1), str(5)
memory usage: 109.6 KB


INSPECTING LOCATION PATTERNS TO CREATE CLEAN LOCATION COLUMN

In [142]:
print("Unique_locatins:",all_jobs["location"].nunique())

Unique_locatins: 452


In [143]:
all_jobs["location"].value_counts().head(50)

location
New York, New York, USA                                90
San Francisco, CA • New York, NY • United States       82
Remote - US                                            66
Tokyo, Japan                                           53
Singapore                                              48
Paris, France                                          39
Bangalore, India                                       39
Remote, United States                                  37
Remote, Canada; Remote, United States                  32
Dublin                                                 30
Bengaluru                                              30
London                                                 27
Americas Remote                                        24
United Kingdom                                         23
Boston, Massachusetts, USA; New York, New York, USA    23
Mexico City                                            21
Remote, United Kingdom                                 20
N/A  

CREATING NEW CLEAN LOCATION DATA

In [144]:
all_jobs["location_clean"]=all_jobs["location"]

#Standardization Rules
location_map={
    "Bengalore,India":"Bengaluru,India",

    "London":"London,United Kingdom",
    "London,England":"London,United Kingdom",
    "United Kindgdom":"United Kingdom",

    "Dublin":"Dublin,Ireland",
    "Dublin,Ireland":"Dublin,Ireland",

    "Singapore":"Singapore",

    "N/A":"Unknownn"
}
all_jobs["location_clean"]=(all_jobs["location_clean"].replace(location_map))

INSPECTING REMOTE JOBS 

In [145]:
remote_patterns=[
    "Remote",
    "Americas Remote",
    "EMEA Remote",
    "APAC Remote"
]
all_jobs["is_remote"]=(all_jobs["location_clean"].str.contains("|".join(remote_patterns),case=False,na=False))

In [146]:
all_jobs["is_remote"].value_counts()


is_remote
False    1192
True      559
Name: count, dtype: int64

COMPARING OUR CHANGES WITH PREVIOUS ONES

In [147]:
Comparison=(
    all_jobs[
        ["location","location_clean","is_remote"]
    ].drop_duplicates().head(30)    
)
Comparison

,location,location_clean,is_remote
0,United Kingdom,United Kingdom,False
1,"Berlin, Germany","Berlin, Germany",False
3,"India, Hyderabad","India, Hyderabad",False
4,Bulgaria,Bulgaria,False
5,"India, Delhi NCR","India, Delhi NCR",False
13,Serbia,Serbia,False
14,Lithuania,Lithuania,False
22,"India, Bengaluru","India, Bengaluru",False
49,"Paris, France","Paris, France",False
73,"Berlin, Germany; Bulgaria; India, Bengaluru; I...","Berlin, Germany; Bulgaria; India, Bengaluru; I...",False


CREATING A FUNCTION TO IMPROVE LOCATION CLEANING

In [148]:
def clean_location(location):

    location = str(location).strip()

    if location == "" or location.upper() == "N/A":
        return "Unknown"

    # Split multiple locations
    parts = location.split(";")

    cleaned_parts = []

    for part in parts:

        part = part.strip()

        # Standardize Bangalore/Bengaluru variations
        if "bangalore" in part.lower() or "bengaluru" in part.lower():
            if part.lower().startswith("remote"):
                part = "Remote, Bengaluru, India"
            else:
                part = "Bengaluru, India"

        # Standardize London
        elif part.lower() in ["london", "london, england"]:
            part = "London, United Kingdom"

        # Standardize Dublin
        elif part.lower() == "dublin":
            part = "Dublin, Ireland"

        # Standardize US remote
        elif part.lower() in ["remote - us", "remote, us"]:
            part = "Remote, United States"

        # Standardize Ireland remote
        elif part.lower() == "remote ireland":
            part = "Remote, Ireland"

        cleaned_parts.append(part)

    # Remove duplicates while preserving order
    cleaned_parts = list(dict.fromkeys(cleaned_parts))

    return "; ".join(cleaned_parts)

In [149]:
all_jobs["location_clean"] = all_jobs["location"].apply(clean_location)

In [150]:
print("Original unique locations:", all_jobs["location"].nunique())
print("Clean unique locations:", all_jobs["location_clean"].nunique())

Original unique locations: 452
Clean unique locations: 420


In [151]:
all_jobs[
    all_jobs["location"].str.contains(
        "Bangalore|Bengaluru",
        case=False,
        na=False
    )
][["location", "location_clean"]].drop_duplicates()

,location,location_clean
22,"India, Bengaluru","Bengaluru, India"
73,"Berlin, Germany; Bulgaria; India, Bengaluru; I...","Berlin, Germany; Bulgaria; Bengaluru, India; I..."
85,"Remote, Bangalore","Remote, Bengaluru, India"
92,"Bangalore, India","Bengaluru, India"
255,"Bangalore, India; Remote, Canada; Remote, Isra...","Bengaluru, India; Remote, Canada; Remote, Isra..."
317,"Bengaluru, India","Bengaluru, India"
538,Bengaluru,"Bengaluru, India"
594,Bangalore,"Bengaluru, India"
630,IN-Bengaluru,"Bengaluru, India"
745,Bangalore,"Bengaluru, India"


In [152]:
all_jobs["is_remote"] = all_jobs["location_clean"].str.contains(
    "remote",
    case=False,
    na=False
)

In [153]:
all_jobs["is_remote"].value_counts()

is_remote
False    1192
True      559
Name: count, dtype: int64

CHECKING REMOTE JOBS PERCENTAGE

In [154]:
print(f"Remote Percentage: ",all_jobs["is_remote"].mean()*100)

Remote Percentage:  31.924614505996573


CHECKING REMOTE JOBS BY COMPANY

In [155]:
all_jobs.groupby("company")["is_remote"].mean().sort_values(ascending=False) * 100

company
Twilio             100.000000
Customer.io        100.000000
GitLab              85.714286
Webflow             82.758621
Karbon              53.333333
BigID               28.571429
Stripe              16.013072
Datadog             12.837838
Careers at Tide      0.000000
Figma                0.000000
Name: is_remote, dtype: float64

## Final Feature Engineering

This section contains the cleaned, reproducible feature-engineering steps completed during the project.


In [156]:
# Basic derived features
all_jobs["published_year"] = all_jobs["published"].dt.year
all_jobs["description_length"] = all_jobs["description"].str.len()
all_jobs["description_word_count"] = all_jobs["description"].str.split().str.len()
all_jobs["title_length"] = all_jobs["title"].str.len()


### AI vocabulary and title/description flags


In [157]:
ai_keywords = [
    "artificial intelligence",
    "generative ai",
    "machine learning",
    "deep learning",
    "large language model",
    "llm",
    "llms",
    "genai",
    "ai agents",
    "ai agent",
    "computer vision",
    "natural language processing",
    "nlp",
    "reinforcement learning",
    "prompt engineering",
    "foundation model",
    "transformer model"
]

ai_keywords_title = (
    r"\bAI\b"
    r"|Artificial Intelligence"
    r"|Machine Learning"
    r"|\bML\b"
    r"|Data Scientist"
    r"|Data Science"
    r"|LLM"
    r"|Generative AI"
)

all_jobs["has_ai_in_title"] = all_jobs["title"].str.contains(
    ai_keywords_title,
    case=False,
    na=False,
    regex=True
)

pattern = "|".join(re.escape(k) for k in ai_keywords)

all_jobs["has_ai_in_description"] = all_jobs["description"].str.contains(
    pattern,
    case=False,
    na=False,
    regex=True
)

def find_ai_keywords(text):
    text = str(text).lower()
    return [
        keyword for keyword in ai_keywords
        if keyword.lower() in text
    ]

all_jobs["ai_keywords_found"] = all_jobs["description"].apply(find_ai_keywords)

all_jobs["ai_keyword_count"] = all_jobs["ai_keywords_found"].str.len()


In [158]:
from collections import Counter

keyword_counts = Counter(
    keyword
    for keywords in all_jobs["ai_keywords_found"]
    for keyword in keywords
)

ai_keyword_counts = (
    pd.DataFrame(
        keyword_counts.items(),
        columns=["ai_keyword", "job_count"]
    )
    .sort_values("job_count", ascending=False)
    .reset_index(drop=True)
)

ai_keyword_counts


,ai_keyword,job_count
0,artificial intelligence,194
1,llm,131
2,ai agent,78
3,machine learning,74
4,ai agents,67
5,llms,52
6,generative ai,29
7,prompt engineering,23
8,large language model,20
9,foundation model,16


In [159]:
ai_categories = {
    "Artificial Intelligence": ["artificial intelligence"],
    "LLM": ["llm", "llms", "large language model"],
    "Machine Learning": ["machine learning"],
    "AI Agents": ["ai agent", "ai agents"],
    "Generative AI": ["generative ai", "genai"],
    "Prompt Engineering": ["prompt engineering"],
    "Foundation Models": ["foundation model"],
    "Deep Learning": ["deep learning"],
    "NLP": ["nlp", "natural language processing"],
    "Reinforcement Learning": ["reinforcement learning"],
    "Computer Vision": ["computer vision"],
    "Transformer Models": ["transformer model"]
}

def find_ai_categories(keywords):
    keywords = [k.lower() for k in keywords]
    categories = []

    for category, terms in ai_categories.items():
        if any(term.lower() in keywords for term in terms):
            categories.append(category)

    return categories

all_jobs["ai_categories_found"] = all_jobs["ai_keywords_found"].apply(
    find_ai_categories
)

all_jobs["ai_category_count"] = all_jobs["ai_categories_found"].str.len()


### AI sentences and boilerplate cleanup


In [160]:
ai_sentence_patterns = [
    r"\bartificial intelligence\b",
    r"\bmachine learning\b",
    r"\bllm\b",
    r"\bllms\b",
    r"\blarge language model\b",
    r"\blarge language models\b",
    r"\bgenerative ai\b",
    r"\bgenai\b",
    r"\bai agent\b",
    r"\bai agents\b",
    r"\bagentic\b",
    r"\bprompt engineering\b",
    r"\bfoundation model\b",
    r"\bfoundation models\b",
    r"\bdeep learning\b",
    r"\bnlp\b",
    r"\bnatural language processing\b",
    r"\breinforcement learning\b",
    r"\bcomputer vision\b",
    r"\btransformer model\b",
    r"\btransformer models\b",
    r"\brag\b",
    r"\bretrieval augmented generation\b",
    r"\bAI\b"
]

ai_sentence_pattern = re.compile(
    "|".join(ai_sentence_patterns),
    re.IGNORECASE
)

def extract_ai_sentences(description):
    sentences = re.split(r"(?<=[.!?])\s+", str(description))

    return [
        sentence.strip()
        for sentence in sentences
        if ai_sentence_pattern.search(sentence)
    ]

all_jobs["ai_sentences"] = all_jobs["description"].apply(
    extract_ai_sentences
)

def clean_ai_sentences(sentences):
    cleaned = []

    for sentence in sentences:
        sentence_lower = sentence.lower()

        # Remove the repeated hiring-process boilerplate identified
        # during manual validation.
        if (
            "use artificial intelligence" in sentence_lower
            and "hiring process" in sentence_lower
        ):
            continue

        cleaned.append(sentence)

    return cleaned

all_jobs["ai_sentences_clean"] = all_jobs["ai_sentences"].apply(
    clean_ai_sentences
)

all_jobs["has_ai_after_boilerplate"] = (
    all_jobs["ai_sentences_clean"].str.len() > 0
)

all_jobs["ai_sentence_count"] = (
    all_jobs["ai_sentences_clean"].str.len()
)


### Seniority classification


In [161]:
def classify_seniority(title):
    title = str(title).lower()

    if re.search(r"\b(chief|c-suite|cmo|cto|cio|cfo|coo)\b", title):
        return "VP/Executive"

    if re.search(r"\b(vp|vice president)\b", title):
        return "VP/Executive"

    if re.search(r"\b(director)\b", title):
        return "Director"

    if re.search(r"\b(head of|head,)\b", title):
        return "Director"

    if re.search(r"\b(manager|mgr)\b", title):
        return "Manager"

    if re.search(r"\b(lead|principal|staff)\b", title):
        return "Lead"

    if re.search(r"\b(senior|sr\.?)\b", title):
        return "Senior"

    if re.search(
        r"\b(junior|jr\.?|entry[- ]level|intern|internship|"
        r"graduate|new grad)\b",
        title
    ):
        return "Entry/Junior"

    return "Mid"

all_jobs["seniority"] = all_jobs["title"].apply(classify_seniority)


### Final role classification


In [162]:
def classify_role_type_v4(title):
    title = str(title).lower().strip()

    if re.search(
        r"\b(security|infosec|information security|appsec|cybersecurity|"
        r"threat detection|incident response|security operations)\b",
        title
    ):
        return "Security"

    if re.search(
        r"\b(data scientist|data science|machine learning|ml engineer|"
        r"ai engineer|artificial intelligence|applied scientist|"
        r"research scientist|research engineer|data and ai specialist|"
        r"generative ai|ai specialist|ai transformation|ai strategy)\b",
        title
    ):
        return "AI / Data Science"

    if re.search(
        r"\b(data analyst|data analytics|analytics analyst|"
        r"business intelligence|bi analyst|business analyst|"
        r"people analytics|product analytics|risk analytics|"
        r"analytics,|analytics manager|analytics lead|"
        r"aml analytics|fraud analytics|revenue analytics|"
        r"pricing analytics)\b",
        title
    ):
        return "Data / Analytics"

    if re.search(
        r"\b(software engineer|software developer|backend|frontend|"
        r"full stack|fullstack|developer|devops|sre|"
        r"site reliability|platform engineer|systems engineer|"
        r"mobile engineer|engineering manager|head of engineering|"
        r"director of engineering|staff engineer|principal engineer|"
        r"senior engineer|forward deployed engineer|field cto|cto|"
        r"technology|compute|flutter|rpa engineer|"
        r"automation engineer|robotics engineer|"
        r"director of mobile|mobile and web platform)\b",
        title
    ):
        return "Engineering / Technology"

    if re.search(
        r"\b(it engineer|it analyst|it manager|it support|"
        r"infrastructure engineer|infrastructure manager|"
        r"cloud engineer|network engineer|systems administrator|"
        r"technical infrastructure|it sox|sox pmo)\b",
        title
    ):
        return "IT / Infrastructure"

    if re.search(
        r"\b(solutions architect|solution architect|professional services|"
        r"presales|pre-sales|implementation consultant|"
        r"solutions consultant|technical consultant|"
        r"professional services engineer|implementation specialist|"
        r"solutions architecture|manager, solutions)\b",
        title
    ):
        return "Professional Services / Solutions"

    if re.search(
        r"\b(product manager|product management|product designer|"
        r"product analyst|product operations|product lead)\b",
        title
    ):
        return "Product"

    if re.search(
        r"\b(sales|account executive|account manager|sales engineer|"
        r"sales development|business development|"
        r"business development representative|partner manager|"
        r"partner development|partner enablement|partnerships|"
        r"channel manager|channels|deal desk|renewals manager|renewals)\b",
        title
    ):
        return "Sales / Business Development"

    if re.search(
        r"\b(marketing|brand|content|communications|social media|growth)\b",
        title
    ):
        return "Marketing"

    if re.search(
        r"\b(finance|financial|accounting|accountant|treasury|tax|"
        r"controller|commercial finance|revenue accounting|collections|"
        r"fraud investigator|payments fraud|aml|financial crimes|"
        r"internal auditor|audit|sec analyst|securities analyst|"
        r"financial analyst)\b",
        title
    ):
        return "Finance / Accounting"

    if re.search(
        r"\b(human resources|hr|recruiting|recruiter|people|talent|sourcer)\b",
        title
    ):
        return "HR / Recruiting"

    if re.search(
        r"\b(customer success|customer support|technical support|"
        r"support engineer|member support|technical account management|"
        r"technical account manager|director, support|support director|"
        r"support engineering|partner success)\b",
        title
    ):
        return "Customer Support / Success"

    if re.search(
        r"\b(legal|counsel|attorney|compliance|privacy)\b",
        title
    ):
        return "Legal / Compliance"

    if re.search(
        r"\b(design|designer|ux|ui|user experience)\b",
        title
    ):
        return "Design"

    if re.search(
        r"\b(operations|operations specialist|program manager|"
        r"project manager|enablement ops|renewal ops|risk operations|"
        r"sanctions ops|fraud ops|procurement|purchasing)\b",
        title
    ):
        return "Operations"

    return "Other"

all_jobs["role_type"] = all_jobs["title"].apply(
    classify_role_type_v4
)


### Country classification


In [163]:
def extract_country(location):
    location = str(location).lower().strip()

    if location in ["unknown", "location", "nan", "none", ""]:
        return "Unknown"

    if re.search(
        r"\b(united states|usa|u\.s\.|u\.s\. remote|"
        r"new york|nyc|chicago|san francisco|seattle|"
        r"south san francisco|california|illinois|washington|"
        r"atlanta|boston|austin|denver|los angeles)\b",
        location
    ):
        return "United States"

    if re.search(
        r"\b(india|bengaluru|bangalore|mumbai|delhi|hyderabad|pune|chennai)\b",
        location
    ):
        return "India"

    if re.search(
        r"\b(united kingdom|uk|england|london|manchester|edinburgh)\b",
        location
    ):
        return "United Kingdom"

    if re.search(
        r"\b(canada|toronto|vancouver|montreal|bc|on)\b",
        location
    ):
        return "Canada"

    if re.search(r"\b(germany|berlin|munich|hamburg)\b", location):
        return "Germany"

    if re.search(r"\b(france|paris|lyon)\b", location):
        return "France"

    if re.search(r"\b(netherlands|amsterdam)\b", location):
        return "Netherlands"

    if re.search(r"\b(australia|sydney|melbourne|brisbane)\b", location):
        return "Australia"

    if "singapore" in location:
        return "Singapore"

    if re.search(r"\b(japan|tokyo|osaka)\b", location):
        return "Japan"

    if re.search(r"\b(brazil|são paulo|sao paulo|rio de janeiro)\b", location):
        return "Brazil"

    if re.search(r"\b(spain|madrid|barcelona)\b", location):
        return "Spain"

    if re.search(r"\b(ireland|dublin)\b", location):
        return "Ireland"

    if re.search(r"\b(mexico|mexico city)\b", location):
        return "Mexico"

    if re.search(r"\b(colombia|bogota|medellin)\b", location):
        return "Colombia"

    if re.search(r"\b(south korea|seoul|korea)\b", location):
        return "South Korea"

    if re.search(r"\b(israel|tel aviv|jerusalem)\b", location):
        return "Israel"

    if re.search(r"\b(bulgaria|sofia)\b", location):
        return "Bulgaria"

    if re.search(r"\b(lithuania|vilnius)\b", location):
        return "Lithuania"

    if re.search(r"\b(serbia|belgrade)\b", location):
        return "Serbia"

    if re.search(r"\b(poland|warsaw|krakow)\b", location):
        return "Poland"

    if re.search(r"\b(portugal|lisbon)\b", location):
        return "Portugal"

    if re.search(r"\b(estonia|tallinn)\b", location):
        return "Estonia"

    if re.search(r"\b(argentina|buenos aires)\b", location):
        return "Argentina"

    if re.search(r"\b(indonesia|jakarta)\b", location):
        return "Indonesia"

    if re.search(r"\b(saudi arabia|riyadh|ksa)\b", location):
        return "Saudi Arabia"

    if re.search(r"\b(united arab emirates|uae|dubai|abu dhabi)\b", location):
        return "United Arab Emirates"

    if re.search(r"\b(turkey|istanbul)\b", location):
        return "Turkey"

    if re.search(r"\b(romania|bucharest)\b", location):
        return "Romania"

    if re.search(r"\b(italy|milan|rome)\b", location):
        return "Italy"

    # Additional country/city mappings found during validation
    if re.search(r"\b(us|us-remote|us remote|remote in the us|"
                 r"us-sf|us-ny|us-atl|us-chi|us-rem|"
                 r"amer - us|us remote national|us-sf-hq)\b", location):
        return "United States"

    if re.search(r"\b(sf|chi|atl|ny|nyc|sea|minnesota|ohio|"
                 r"dc|maryland|virginia)\b", location):
        return "United States"

    if re.search(r"\b(taiwan|taipei)\b", location):
        return "Taiwan"

    if "luxembourg" in location:
        return "Luxembourg"

    if re.search(r"\b(new zealand|auckland)\b", location):
        return "New Zealand"

    if re.search(r"\b(thailand|bangkok)\b", location):
        return "Thailand"

    if re.search(r"\b(denmark|copenhagen)\b", location):
        return "Denmark"

    if re.search(r"\b(sweden|stockholm)\b", location):
        return "Sweden"

    if re.search(r"\b(chile)\b", location):
        return "Chile"

    return "Other"

all_jobs["country"] = all_jobs["location_clean"].apply(extract_country)


### Location type classification


In [164]:
def classify_location_type(location):
    location = str(location).lower().strip()

    if location in ["unknown", "location", "nan", "none", ""]:
        return "Unknown"

    if re.search(
        r"\b(americas remote|emea remote|amer remote|"
        r"remote, north america|north america remote|"
        r"remote, americas)\b",
        location
    ):
        return "Regional Remote"

    if (
        ";" in location
        or " or " in location
        or " and " in location
        or " • " in location
    ):
        return "Multiple Locations"

    if "remote" in location:
        return "Remote"

    return "Location-based"

all_jobs["location_type"] = all_jobs["location_clean"].apply(
    classify_location_type
)


### AI relevance status

The AI relevance classification was deliberately **not frozen** yet. Manual validation found false positives (for example, `LLM` can mean Master of Laws) and cases where AI-focused titles were under-classified. The cleaned AI sentence evidence above is therefore retained as the auditable basis for the next classification pass.


In [165]:
# Current AI baseline before final relevance classification
ai_baseline = (
    all_jobs[["has_ai_in_title", "has_ai_in_description"]]
    .value_counts()
    .rename("job_count")
    .reset_index()
)

ai_baseline


,has_ai_in_title,has_ai_in_description,job_count
0,False,False,1312
1,False,True,328
2,True,True,83
3,True,False,28


## Final Dataset Audit


In [166]:
print("Rows:", len(all_jobs))
print("Columns:", len(all_jobs.columns))
print("Unique job IDs:", all_jobs["job_id"].nunique())
print("Duplicate job IDs:", all_jobs["job_id"].duplicated().sum())

all_jobs.info()


Rows: 1751
Columns: 28
Unique job IDs: 1751
Duplicate job IDs: 0
<class 'pandas.DataFrame'>
RangeIndex: 1751 entries, 0 to 1750
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   job_id                    1751 non-null   int64              
 1   title                     1751 non-null   str                
 2   company                   1751 non-null   str                
 3   location                  1751 non-null   str                
 4   url                       1751 non-null   str                
 5   published                 1751 non-null   datetime64[us, UTC]
 6   updated                   1751 non-null   datetime64[us, UTC]
 7   description               1751 non-null   str                
 8   location_clean            1751 non-null   str                
 9   is_remote                 1751 non-null   bool               
 10  published_year            1751

In [167]:
all_jobs[
    [
        "published_year",
        "description_length",
        "description_word_count",
        "title_length",
        "ai_keyword_count",
        "ai_category_count",
        "ai_sentence_count"
    ]
].describe()


,published_year,description_length,description_word_count,title_length,ai_keyword_count,ai_category_count,ai_sentence_count
count,1751.000000,1751.000000,1751.000000,1751.000000,1751.000000,1751.000000,1751.000000
mean,2025.843518,5906.550543,852.396916,37.251856,0.415191,0.341519,3.033124
std,0.536190,1950.496444,283.862048,12.061223,0.959511,0.769331,3.292596
min,2019.000000,2227.000000,268.000000,10.000000,0.000000,0.000000,0.000000
25%,2026.000000,4400.000000,635.000000,28.000000,0.000000,0.000000,0.000000
50%,2026.000000,5579.000000,801.000000,36.000000,0.000000,0.000000,2.000000
75%,2026.000000,7400.500000,1067.000000,45.000000,0.000000,0.000000,5.000000
max,2026.000000,14791.000000,1964.000000,89.000000,10.000000,9.000000,30.000000


In [168]:
print("Role distribution:")
display(all_jobs["role_type"].value_counts())

print("\nCountry distribution:")
display(all_jobs["country"].value_counts())

print("\nLocation type distribution:")
display(all_jobs["location_type"].value_counts())

print("\nSeniority distribution:")
display(all_jobs["seniority"].value_counts())


Role distribution:


role_type
Sales / Business Development         435
Engineering / Technology             331
Other                                219
Operations                           104
Product                              102
Professional Services / Solutions     92
Customer Support / Success            88
Marketing                             74
Security                              72
Finance / Accounting                  68
HR / Recruiting                       49
AI / Data Science                     47
Legal / Compliance                    25
Design                                20
Data / Analytics                      15
IT / Infrastructure                   10
Name: count, dtype: int64


Country distribution:


country
United States           808
United Kingdom          154
India                   126
Singapore                75
Ireland                  64
Japan                    64
Canada                   62
France                   52
Other                    40
Australia                37
Mexico                   36
Germany                  31
Brazil                   24
Unknown                  24
Bulgaria                 18
Colombia                 16
Netherlands              15
Spain                    13
South Korea              12
Israel                   12
Poland                   10
Lithuania                 7
Portugal                  6
Serbia                    5
Saudi Arabia              5
Argentina                 5
Italy                     4
Estonia                   4
Indonesia                 4
United Arab Emirates      3
Taiwan                    3
Romania                   3
Turkey                    2
Denmark                   2
Luxembourg                1
New Zealand 


Location type distribution:


location_type
Location-based        971
Remote                389
Multiple Locations    337
Regional Remote        30
Unknown                24
Name: count, dtype: int64


Seniority distribution:


seniority
Mid             770
Manager         516
Senior          193
Lead            174
Director         71
Entry/Junior     18
VP/Executive      9
Name: count, dtype: int64

In [169]:
all_jobs[[
    "location",
    "location_clean",
    "country",
    "location_type",
    "is_remote"
]].head(20)

,location,location_clean,country,location_type,is_remote
0,United Kingdom,United Kingdom,United Kingdom,Location-based,False
1,"Berlin, Germany","Berlin, Germany",Germany,Location-based,False
2,United Kingdom,United Kingdom,United Kingdom,Location-based,False
3,"India, Hyderabad","India, Hyderabad",India,Location-based,False
4,Bulgaria,Bulgaria,Bulgaria,Location-based,False
5,"India, Delhi NCR","India, Delhi NCR",India,Location-based,False
6,"India, Delhi NCR","India, Delhi NCR",India,Location-based,False
7,"Berlin, Germany","Berlin, Germany",Germany,Location-based,False
8,United Kingdom,United Kingdom,United Kingdom,Location-based,False
9,Bulgaria,Bulgaria,Bulgaria,Location-based,False


In [170]:
all_jobs[
    all_jobs["has_ai_after_boilerplate"] == True
][[
    "company",
    "title",
    "ai_keywords_found",
    "ai_sentences_clean"
]].head(20)

,company,title,ai_keywords_found,ai_sentences_clean
0,Careers at Tide,Account Executive,[],[Tide leverages AI to enhance our hiring exper...
1,Careers at Tide,Account Executive – SME Credit,[],[Tide leverages AI to enhance our hiring exper...
2,Careers at Tide,Asset Finance Specialist,[],[Tide leverages AI to enhance our hiring exper...
3,Careers at Tide,Business Development Executive - French Speaking,[],[Tide leverages AI to enhance our hiring exper...
4,Careers at Tide,Business Development Executive - French Speaking,[],[Tide leverages AI to enhance our hiring exper...
5,Careers at Tide,Business Development Specialist (International...,[],[Tide leverages AI to enhance our hiring exper...
6,Careers at Tide,Business Development Specialist (Outbound- Int...,[],[Tide leverages AI to enhance our hiring exper...
7,Careers at Tide,Business Operations Manager,[],[Tide leverages AI to enhance our hiring exper...
8,Careers at Tide,Commercial Services - UK Country Commercial Lead,[],[Tide leverages AI to enhance our hiring exper...
9,Careers at Tide,Customer Support Executive with German (Onboar...,[],[Tide leverages AI to enhance our hiring exper...


In [171]:
all_jobs[
    all_jobs["has_ai_after_boilerplate"] == True
][
    [
        "company",
        "title",
        "ai_keywords_found",
        "ai_sentences_clean"
    ]
].head(30)

,company,title,ai_keywords_found,ai_sentences_clean
0,Careers at Tide,Account Executive,[],[Tide leverages AI to enhance our hiring exper...
1,Careers at Tide,Account Executive – SME Credit,[],[Tide leverages AI to enhance our hiring exper...
2,Careers at Tide,Asset Finance Specialist,[],[Tide leverages AI to enhance our hiring exper...
3,Careers at Tide,Business Development Executive - French Speaking,[],[Tide leverages AI to enhance our hiring exper...
4,Careers at Tide,Business Development Executive - French Speaking,[],[Tide leverages AI to enhance our hiring exper...
5,Careers at Tide,Business Development Specialist (International...,[],[Tide leverages AI to enhance our hiring exper...
6,Careers at Tide,Business Development Specialist (Outbound- Int...,[],[Tide leverages AI to enhance our hiring exper...
7,Careers at Tide,Business Operations Manager,[],[Tide leverages AI to enhance our hiring exper...
8,Careers at Tide,Commercial Services - UK Country Commercial Lead,[],[Tide leverages AI to enhance our hiring exper...
9,Careers at Tide,Customer Support Executive with German (Onboar...,[],[Tide leverages AI to enhance our hiring exper...


In [172]:
import re

def classify_ai_relevance(row):
    title = str(row["title"]).lower()
    description = str(row["description"]).lower()

    # Strong AI signals in the job title
    ai_title_patterns = [
        r"\bartificial intelligence\b",
        r"\bai engineer\b",
        r"\bai scientist\b",
        r"\bai specialist\b",
        r"\bai strategy\b",
        r"\bai transformation\b",
        r"\bai platform\b",
        r"\bai platforms\b",
        r"\bai infrastructure\b",
        r"\bai product\b",
        r"\bai products\b",
        r"\bai & platforms\b",
        r"\bmachine learning\b",
        r"\bml engineer\b",
        r"\bml scientist\b",
        r"\bllm\b",
        r"\bllms\b",
        r"\blarge language model\b",
        r"\bgenerative ai\b",
        r"\bgenai\b",
        r"\bagentic\b",
        r"\bagentic ai\b",
        r"\bai agents?\b",
        r"\bdeep learning\b",
        r"\bnlp\b",
        r"\bnatural language processing\b",
        r"\bcomputer vision\b",
        r"\bfoundation models?\b"
    ]

    # Strong AI signals in responsibilities / requirements
    ai_responsibility_patterns = [
        r"\bmachine learning\b",
        r"\bdeep learning\b",
        r"\bllms?\b",
        r"\blarge language models?\b",
        r"\bgenerative ai\b",
        r"\bgenai\b",
        r"\bartificial intelligence\b",
        r"\bai agents?\b",
        r"\bagentic ai\b",
        r"\bagentic\b",
        r"\bprompt engineering\b",
        r"\bfoundation models?\b",
        r"\bnatural language processing\b",
        r"\bnlp\b",
        r"\bcomputer vision\b",
        r"\bretrieval augmented generation\b",
        r"\brag pipelines?\b",
        r"\bembeddings?\b"
    ]

    # Generic company/recruiting statements that should NOT make
    # an otherwise unrelated job an AI-related job.
    boilerplate_patterns = [
        r"we use artificial intelligence.*hiring process",
        r"we use ai.*hiring process",
        r"leverages ai to enhance our hiring",
        r"ai.*recruitment process",
        r"ai policy",
        r"ai guidelines",
        r"privacy and ai guidelines"
    ]

    # Remove known generic boilerplate from description
    cleaned_description = description

    for pattern in boilerplate_patterns:
        cleaned_description = re.sub(
            pattern,
            " ",
            cleaned_description,
            flags=re.IGNORECASE
        )

    # Check strong AI title signal
    title_ai = any(
        re.search(pattern, title, re.IGNORECASE)
        for pattern in ai_title_patterns
    )

    # Count meaningful AI signals in the cleaned description
    description_matches = [
        pattern
        for pattern in ai_responsibility_patterns
        if re.search(pattern, cleaned_description, re.IGNORECASE)
    ]

    description_ai = len(description_matches) > 0

    # Strong title + AI-related description
    if title_ai and description_ai:
        return "AI Required / Responsibility"

    # Strong AI-focused title by itself
    if title_ai:
        return "AI Required / Responsibility"

    # Description contains multiple meaningful AI concepts
    if len(description_matches) >= 2:
        return "AI Required / Responsibility"

    # One meaningful AI requirement
    if len(description_matches) == 1:
        return "AI Mention Only"

    return "No AI Mention"


all_jobs["ai_relevance"] = all_jobs.apply(
    classify_ai_relevance,
    axis=1
)

In [173]:
all_jobs["ai_relevance"].value_counts()


ai_relevance
No AI Mention                   1375
AI Mention Only                  202
AI Required / Responsibility     174
Name: count, dtype: int64

In [174]:
all_jobs[
    all_jobs["ai_relevance"] != "No AI Mention"
][[
    "company",
    "title",
    "ai_relevance",
    "ai_keywords_found",
    "ai_sentences_clean"
]].head(50)

,company,title,ai_relevance,ai_keywords_found,ai_sentences_clean
70,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm],"[Tide facts: Tide is available for UK, Indian,..."
71,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm],"[Tide facts: Tide is available for UK, Indian,..."
72,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm],"[Tide facts: Tide is available for UK, Indian,..."
73,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm],[ABOUT THE ROLE: Tide is hiring a Senior Staff...
74,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm],"[Tide facts: Tide is available for UK, Indian,..."
75,Careers at Tide,"Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
76,Careers at Tide,"Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
77,Careers at Tide,"Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
78,Careers at Tide,"Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
79,Careers at Tide,"Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[generative ai],"[Tide facts: Tide is available for UK, Indian,..."


In [175]:
ai_check = all_jobs[
    all_jobs["ai_relevance"] == "AI Mention Only"
][[
    "company",
    "title",
    "ai_keywords_found",
    "ai_sentences_clean"
]]

ai_check.head(100)

,company,title,ai_keywords_found,ai_sentences_clean
93,GitLab,"Backend Engineer (Ruby), AI Engineering: Agent...","[ai agents, ai agent]",[The same principles built into our products a...
112,GitLab,"Director of Engineering, Organizations & Cells",[],[The same principles built into our products a...
116,GitLab,"Director, Support (Bengaluru)",[artificial intelligence],[The same principles built into our products a...
121,GitLab,"Engineering Manager, Agent Foundations: Agent ...","[ai agents, ai agent]",[The same principles built into our products a...
123,GitLab,"Engineering Manager, Cell Infrastructure",[],[The same principles built into our products a...
...,...,...,...,...
839,Stripe,"Product Manager, Link - Local Payment Methods","[ai agents, ai agent]",[About the team The Link team builds the premi...
842,Stripe,"Product Manager, Sail Core",[],[Responsibilities Own the product strategy and...
857,Stripe,Product Strategy & Operations - Global Product,[generative ai],[Experience in leveraging Generative AI tools ...
865,Stripe,"Program Manager, Performance and Talent Planning",[ai agent],[A Day in the Life You start your day by revie...


In [176]:
all_jobs[
    all_jobs["ai_relevance"] == "AI Required / Responsibility"
][[
    "company",
    "title",
    "ai_keywords_found",
    "ai_sentences_clean"
]].head(50)

,company,title,ai_keywords_found,ai_sentences_clean
70,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],"[Tide facts: Tide is available for UK, Indian,..."
71,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],"[Tide facts: Tide is available for UK, Indian,..."
72,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],"[Tide facts: Tide is available for UK, Indian,..."
73,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],[ABOUT THE ROLE: Tide is hiring a Senior Staff...
74,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],"[Tide facts: Tide is available for UK, Indian,..."
75,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
76,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
77,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
78,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
79,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."


In [177]:
required_check = all_jobs[
    all_jobs["ai_relevance"] == "AI Required / Responsibility"
][[
    "company",
    "title",
    "ai_keywords_found",
    "ai_sentences_clean"
]]

required_check.head(50)

,company,title,ai_keywords_found,ai_sentences_clean
70,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],"[Tide facts: Tide is available for UK, Indian,..."
71,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],"[Tide facts: Tide is available for UK, Indian,..."
72,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],"[Tide facts: Tide is available for UK, Indian,..."
73,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],[ABOUT THE ROLE: Tide is hiring a Senior Staff...
74,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm],"[Tide facts: Tide is available for UK, Indian,..."
75,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
76,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
77,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
78,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."
79,Careers at Tide,"Staff Software Engineer, Agentic Platform",[generative ai],"[Tide facts: Tide is available for UK, Indian,..."


In [178]:
import re

# ---------------------------------------------------------
# FINAL AI RELEVANCE CLASSIFIER
# ---------------------------------------------------------

# Strong AI signals in job titles
AI_TITLE_STRONG = [
    r"\bai engineer\b",
    r"\bai scientist\b",
    r"\bai specialist\b",
    r"\bai architect\b",
    r"\bai researcher\b",
    r"\bmachine learning engineer\b",
    r"\bmachine learning scientist\b",
    r"\bml engineer\b",
    r"\bml scientist\b",
    r"\bdata scientist\b",
    r"\bai product\b",
    r"\bai products\b",
    r"\bai platform\b",
    r"\bai platforms\b",
    r"\bai infrastructure\b",
    r"\bai transformation\b",
    r"\bai strategy\b",
    r"\bagentic ai\b",
    r"\bai agents?\b",
    r"\bagentic platform\b",
    r"\bagentic engineering\b",
    r"\bagentic sdlc\b",
    r"\bai engineering\b",
    r"\bai & platforms\b",
    r"\bai\b.*\b(sme|specialist|owner|lead)\b",
]

# AI-related technologies / concepts
AI_TECH_PATTERNS = [
    r"\bmachine learning\b",
    r"\bdeep learning\b",
    r"\blarge language models?\b",
    r"\bgenerative ai\b",
    r"\bgenai\b",
    r"\bai agents?\b",
    r"\bagentic ai\b",
    r"\bprompt engineering\b",
    r"\bfoundation models?\b",
    r"\bnatural language processing\b",
    r"\bnlp\b",
    r"\bcomputer vision\b",
    r"\bretrieval augmented generation\b",
    r"\brag pipelines?\b",
    r"\bembeddings?\b",
    r"\btransformer models?\b",
]

# Explicit responsibility / work involving AI
AI_RESPONSIBILITY_PATTERNS = [
    r"\bbuild\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\bdevelop\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\bdesign\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\bengineer\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\bdeveloping\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\bbuilding\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\bwork with\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\bworking with\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\bproduction\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\bdeploy\b.{0,100}\b(ai|machine learning|ml|llm|generative ai|genai)\b",
    r"\btraining\b.{0,100}\b(machine learning|ml|deep learning)\b",
    r"\bmodel\b.{0,100}\b(training|inference|evaluation|deployment)\b",
    r"\bllm\b.{0,100}\b(application|applications|system|systems|pipeline|pipelines)\b",
]

# Preferred / nice-to-have AI skills
AI_PREFERRED_PATTERNS = [
    r"\bai\b.{0,80}\b(preferred|preferred qualification|nice to have|nice-to-have)\b",
    r"\b(machine learning|ml|llm|generative ai|genai)\b.{0,80}\b(preferred|preferred qualification|nice to have|nice-to-have)\b",
    r"\b(preferred|preferred qualification|nice to have|nice-to-have)\b.{0,80}\b(machine learning|ml|llm|generative ai|genai)\b",
    r"\bexperience with\b.{0,80}\b(machine learning|ml|llm|generative ai|genai)\b",
]

# Generic company / recruiting AI statements that should NOT
# be treated as evidence that the job itself is an AI job.
BOILERPLATE_PATTERNS = [
    r"we use artificial intelligence.*?(?:\.|$)",
    r"we use ai.*?(?:\.|$)",
    r"leverages ai to enhance our hiring.*?(?:\.|$)",
    r"ai.*?hiring process.*?(?:\.|$)",
    r"ai.*?recruitment process.*?(?:\.|$)",
    r"ai policy.*?(?:\.|$)",
    r"ai guidelines.*?(?:\.|$)",
    r"privacy and ai guidelines.*?(?:\.|$)",
    r"our use of artificial intelligence.*?(?:\.|$)",
]


def remove_ai_boilerplate(text):
    """
    Remove generic company/recruitment AI statements
    before evaluating AI relevance.
    """
    text = str(text).lower()

    for pattern in BOILERPLATE_PATTERNS:
        text = re.sub(
            pattern,
            " ",
            text,
            flags=re.IGNORECASE
        )

    return text


def contains_any_pattern(text, patterns):
    """
    Return True if any regex pattern matches.
    """
    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in patterns
    )


def count_pattern_matches(text, patterns):
    """
    Count unique AI concepts detected in the text.
    """
    count = 0

    for pattern in patterns:
        if re.search(pattern, text, re.IGNORECASE):
            count += 1

    return count


def classify_ai_relevance_final(row):

    title = str(row["title"]).lower().strip()
    description = str(row["description"]).lower()

    # Remove generic company/recruiting AI language
    clean_description = remove_ai_boilerplate(description)

    # -----------------------------------------------------
    # 1. TITLE-BASED AI ROLE
    # -----------------------------------------------------

    title_strong_ai = contains_any_pattern(
        title,
        AI_TITLE_STRONG
    )

    # Explicit AI role focus in title
    if title_strong_ai:
        return "AI Required / Responsibility"

    # Titles containing AI/ML/LLM/GenAI as a clear
    # specialization
    title_specialization = [
        r"\bai\b",
        r"\bmachine learning\b",
        r"\bml\b",
        r"\bllm\b",
        r"\bllms\b",
        r"\bgenerative ai\b",
        r"\bgenai\b",
        r"\bagentic\b",
        r"\bagentic\b",
    ]

    # Avoid treating generic "AI" mentions in unrelated
    # contexts as automatically required.
    if (
        contains_any_pattern(title, title_specialization)
        and re.search(
            r"\b(engineer|scientist|architect|developer|designer|"
            r"manager|director|lead|principal|staff|specialist|"
            r"owner|strategist|product|platform|infrastructure|"
            r"transformation|solutions|researcher)\b",
            title,
            re.IGNORECASE
        )
    ):
        return "AI Required / Responsibility"

    # -----------------------------------------------------
    # 2. DESCRIPTION-BASED AI EVIDENCE
    # -----------------------------------------------------

    responsibility_matches = count_pattern_matches(
        clean_description,
        AI_RESPONSIBILITY_PATTERNS
    )

    technology_matches = count_pattern_matches(
        clean_description,
        AI_TECH_PATTERNS
    )

    preferred_match = contains_any_pattern(
        clean_description,
        AI_PREFERRED_PATTERNS
    )

    # Strong explicit AI responsibility
    if responsibility_matches >= 1:
        return "AI Required / Responsibility"

    # Multiple meaningful AI technologies suggest the
    # technology is relevant to the actual work.
    if technology_matches >= 2:
        return "AI Required / Responsibility"

    # Explicit preferred / nice-to-have AI skills
    if preferred_match:
        return "AI Preferred"

    # -----------------------------------------------------
    # 3. SINGLE MEANINGFUL AI MENTION
    # -----------------------------------------------------

    if technology_matches == 1:
        return "AI Mention Only"

    # -----------------------------------------------------
    # 4. NO MEANINGFUL AI SIGNAL
    # -----------------------------------------------------

    return "No AI Mention"


all_jobs["ai_relevance"] = all_jobs.apply(
    classify_ai_relevance_final,
    axis=1
)

In [179]:
all_jobs["ai_relevance"].value_counts()

ai_relevance
No AI Mention                   1338
AI Required / Responsibility     351
AI Mention Only                   50
AI Preferred                      12
Name: count, dtype: int64

In [180]:
ai_final_check = all_jobs[
    all_jobs["ai_relevance"] != "No AI Mention"
][[
    "company",
    "title",
    "ai_relevance",
    "ai_keywords_found"
]]

ai_final_check.head(100)

,company,title,ai_relevance,ai_keywords_found
70,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm]
71,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm]
72,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm]
73,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm]
74,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",AI Required / Responsibility,[llm]
...,...,...,...,...
276,GitLab,"Staff Backend Engineer, EMEA",AI Required / Responsibility,[]
277,GitLab,"Staff Backend Engineer, EMEA",AI Required / Responsibility,[]
279,GitLab,"Staff Backend Engineer (Go), Events Platform",AI Required / Responsibility,[]
280,GitLab,"Staff Backend Engineer, India",AI Required / Responsibility,[]


In [181]:
all_jobs[
    all_jobs["ai_relevance"] == "AI Required / Responsibility"
][[
    "company",
    "title",
    "ai_keywords_found"
]].head(100)

,company,title,ai_keywords_found
70,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
71,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
72,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
73,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
74,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
...,...,...,...
295,GitLab,"Staff Security Engineer, IAM","[ai agents, ai agent]"
298,GitLab,"Staff Systems Engineer, IT",[]
299,GitLab,"Strategic Account Executive, AI - San Francisco",[foundation model]
302,GitLab,"Strategic Account Executive, Poland",[]


In [182]:
suspicious_ai_required = all_jobs[
    (all_jobs["ai_relevance"] == "AI Required / Responsibility")
    &
    (~all_jobs["title"].str.contains(
        r"\b(ai|artificial intelligence|machine learning|ml|llm|genai|generative ai|agentic)\b",
        case=False,
        regex=True,
        na=False
    ))
][[
    "company",
    "title",
    "ai_keywords_found",
    "ai_sentences_clean"
]]

suspicious_ai_required.head(100)

C:\Users\Shoaib\AppData\Local\Temp\ipykernel_29036\4071316934.py:4: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  (~all_jobs["title"].str.contains(


,company,title,ai_keywords_found,ai_sentences_clean
89,GitLab,Associate Solutions Architect (French or Germa...,[],[The same principles built into our products a...
90,GitLab,Associate Solutions Architect (French or Germa...,[],[The same principles built into our products a...
94,GitLab,Business Development Representative,[],[The same principles built into our products a...
95,GitLab,Business Development Representative,[],[The same principles built into our products a...
102,GitLab,"Commercial Account Executive - Mid-Market, Canada",[],[The same principles built into our products a...
...,...,...,...,...
672,Stripe,"Forward Deployed Engineer, Professional Services",[],[Minimum requirements 5+ years of professional...
673,Stripe,"Forward Deployed Engineer, Professional Services",[],[Minimum requirements 5+ years of professional...
681,Stripe,Fraud Strategist,[machine learning],[Minimum requirements 5+ years of relevant exp...
689,Stripe,"Full Stack Engineer, Growth",[machine learning],[Minimum requirements 2+ years of industry sof...


In [183]:
all_jobs[
    all_jobs["ai_relevance"] == "AI Preferred"
][[
    "company",
    "title",
    "ai_keywords_found",
    "ai_sentences_clean"
]]

,company,title,ai_keywords_found,ai_sentences_clean
219,GitLab,"Senior FP&A Analyst, Cloud Hosting",[],[The same principles built into our products a...
540,Stripe,"Administrative Business Partner, Office of the...",[],[A firm understanding of AI capabilities and a...
586,Stripe,Core Business Recruiter (Fixed Term Contract),[],[Curiosity about emerging AI tools and a pract...
608,Stripe,"Customer Success Manager, Radar",[machine learning],[Minimum requirements 5+ years of enterprise r...
727,Stripe,Indirect Tax Advisory Consultant,[llm],"[LLM in Tax, MSc in Accounting) Experience in ..."
734,Stripe,Internal Audit Data Analytics Lead,[machine learning],[The DA Lead will also help drive the usage of...
751,Stripe,"Manager, Global Sanctions",[],[Minimum requirements At least 10 years of exp...
752,Stripe,"Manager, Global Sanctions",[],[Minimum requirements At least 10 years of exp...
919,Stripe,"Senior Designer, Brand Advertising",[],[Minimum requirements 5+ years of design and a...
992,Stripe,Staff Data Analyst,[],[Minimum requirements 10+ years in Data Analyt...


In [184]:
all_jobs[
    all_jobs["ai_keywords_found"].apply(
        lambda x: "llm" in x if isinstance(x, list) else False
    )
][[
    "company",
    "title",
    "ai_relevance",
    "ai_keywords_found"
]].head(100)

,company,title,ai_relevance,ai_keywords_found
21,Careers at Tide,"Engineering Manager, Member Accession Marketing",No AI Mention,"[llm, llms]"
22,Careers at Tide,"Engineering Manager, Member Accession Marketing",No AI Mention,"[llm, llms]"
23,Careers at Tide,"Engineering Manager, Member Accession Marketing",No AI Mention,"[llm, llms]"
29,Careers at Tide,"Head of Engineering, Commercial Services",No AI Mention,"[llm, llms]"
67,Careers at Tide,"Senior Software Engineer, Flutter",No AI Mention,"[llm, llms]"
...,...,...,...,...
1207,Twilio,Staff Product Manager - Enterprise AI,AI Required / Responsibility,"[artificial intelligence, llm]"
1209,Twilio,Staff Software Engineer,AI Required / Responsibility,"[artificial intelligence, llm, llms, ai agents..."
1211,Twilio,Staff Software Engineer (L4),AI Required / Responsibility,"[artificial intelligence, llm]"
1220,Twilio,Tech Lead /Sr. Principal Engineer (L6),AI Required / Responsibility,"[artificial intelligence, machine learning, ll..."


In [185]:
import re

# =========================================================
# FINAL AI RELEVANCE CLASSIFIER - VERSION 2
# =========================================================

# ---------------------------------------------------------
# 1. DEFINITELY AI-FOCUSED TITLES
# ---------------------------------------------------------

AI_CORE_TITLE_PATTERNS = [
    r"\bai engineer\b",
    r"\bai scientist\b",
    r"\bai researcher\b",
    r"\bai architect\b",
    r"\bai specialist\b",
    r"\bmachine learning engineer\b",
    r"\bmachine learning scientist\b",
    r"\bml engineer\b",
    r"\bml scientist\b",
    r"\bmachine learning researcher\b",
    r"\bai applied scientist\b",
    r"\bapplied scientist\b.*\bai\b",
    r"\bai engineering\b",
    r"\bai platform\b",
    r"\bai infrastructure\b",
    r"\bai transformation\b",
    r"\bagentic platform\b",
    r"\bagentic engineering\b",
    r"\bagentic sdlc\b",
    r"\bai agents?\b",
    r"\bai product\b",
    r"\bai products\b",
    r"\bai strategy\b",
    r"\bai & platforms\b",
    r"\bai software factory\b",
    r"\bai custom models?\b",
]

# AI-related specialization in engineering / product /
# architecture / design / research roles
AI_TITLE_SPECIALIZATION_PATTERNS = [
    r"\b(engineer|developer|architect|scientist|researcher|"
    r"product manager|product designer|designer|specialist|"
    r"platform|infrastructure)\b.*\b(ai|machine learning|"
    r"ml|llm|genai|generative ai|agentic)\b",

    r"\b(ai|machine learning|ml|llm|genai|generative ai|agentic)\b"
    r".*\b(engineer|developer|architect|scientist|researcher|"
    r"product manager|product designer|designer|specialist|"
    r"platform|infrastructure)\b"
]

# ---------------------------------------------------------
# 2. AI TECHNOLOGIES
# ---------------------------------------------------------

AI_TECH_PATTERNS = [
    r"\bmachine learning\b",
    r"\bdeep learning\b",
    r"\blarge language models?\b",
    r"\bgenerative ai\b",
    r"\bgenai\b",
    r"\bai agents?\b",
    r"\bagentic ai\b",
    r"\bprompt engineering\b",
    r"\bfoundation models?\b",
    r"\bnatural language processing\b",
    r"\bnlp\b",
    r"\bcomputer vision\b",
    r"\bretrieval augmented generation\b",
    r"\brag pipelines?\b",
    r"\bembeddings?\b",
    r"\btransformer models?\b",
]

# ---------------------------------------------------------
# 3. REMOVE KNOWN FALSE-POSITIVE / BOILERPLATE TEXT
# ---------------------------------------------------------

def clean_ai_description(text):

    text = str(text).lower()

    boilerplate_patterns = [
        r"we use artificial intelligence.*?(?:\.|$)",
        r"we use ai.*?(?:\.|$)",
        r"leverages ai to enhance our hiring.*?(?:\.|$)",
        r"ai.*?hiring process.*?(?:\.|$)",
        r"ai.*?recruitment process.*?(?:\.|$)",
        r"privacy and ai guidelines.*?(?:\.|$)",
        r"ai policy.*?(?:\.|$)",
        r"ai guidelines.*?(?:\.|$)",
        r"our use of artificial intelligence.*?(?:\.|$)",
    ]

    for pattern in boilerplate_patterns:
        text = re.sub(
            pattern,
            " ",
            text,
            flags=re.IGNORECASE
        )

    # -----------------------------------------------------
    # IMPORTANT:
    # LLM can mean "Master of Laws".
    # Remove legal-degree references before AI analysis.
    # -----------------------------------------------------

    text = re.sub(
        r"\bllm\s+(?:in|degree|qualification|program|course)\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\bmaster of laws\b",
        " ",
        text,
        flags=re.IGNORECASE
    )

    return text


# ---------------------------------------------------------
# 4. CHECK WHETHER AI IS EXPLICITLY PREFERRED
# ---------------------------------------------------------

AI_PREFERRED_PATTERNS = [

    r"\b(ai|artificial intelligence|machine learning|ml|llm|"
    r"generative ai|genai)\b.{0,100}"
    r"\b(preferred|preferred qualification|nice to have|"
    r"nice-to-have|plus)\b",

    r"\b(preferred|preferred qualification|nice to have|"
    r"nice-to-have|plus)\b.{0,100}"
    r"\b(ai|artificial intelligence|machine learning|ml|llm|"
    r"generative ai|genai)\b",
]


# ---------------------------------------------------------
# 5. EXPLICIT AI RESPONSIBILITY
# ---------------------------------------------------------

AI_RESPONSIBILITY_PATTERNS = [

    r"\bresponsible for\b.{0,150}\b(ai|machine learning|ml|"
    r"llm|generative ai|genai|ai agents?)\b",

    r"\bbuild\b.{0,100}\b(ai systems?|ai products?|"
    r"machine learning|ml models?|llm applications?|"
    r"generative ai|ai agents?)\b",

    r"\bdevelop\b.{0,100}\b(ai systems?|ai products?|"
    r"machine learning|ml models?|llm applications?|"
    r"generative ai|ai agents?)\b",

    r"\bdesign\b.{0,100}\b(ai systems?|ai products?|"
    r"machine learning|ml models?|llm applications?|"
    r"generative ai|ai agents?)\b",

    r"\bdeveloping\b.{0,100}\b(ai systems?|ai products?|"
    r"machine learning|ml models?|llm applications?|"
    r"generative ai|ai agents?)\b",

    r"\bbuilding\b.{0,100}\b(ai systems?|ai products?|"
    r"machine learning|ml models?|llm applications?|"
    r"generative ai|ai agents?)\b",

    r"\bdeploy\b.{0,100}\b(ai models?|machine learning|"
    r"ml models?|llm|generative ai|ai agents?)\b",

    r"\btrain\b.{0,100}\b(machine learning|ml models?|"
    r"deep learning|ai models?)\b",

    r"\bevaluate\b.{0,100}\b(llm|ai models?|machine learning|"
    r"generative ai)\b",

    r"\bwork with\b.{0,100}\b(llm|large language models?|"
    r"machine learning|generative ai|ai agents?)\b",
]


# ---------------------------------------------------------
# 6. GENERIC AI USAGE
# ---------------------------------------------------------

AI_USAGE_PATTERNS = [
    r"\bexperience incorporating ai\b",
    r"\buse ai\b",
    r"\busing ai\b",
    r"\bai capabilities\b",
    r"\bai tools\b",
    r"\bai technology\b",
    r"\bai technologies\b",
]


def matches_any(text, patterns):

    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in patterns
    )


def count_ai_technologies(text):

    return sum(
        bool(re.search(pattern, text, re.IGNORECASE))
        for pattern in AI_TECH_PATTERNS
    )


# ---------------------------------------------------------
# 7. FINAL CLASSIFICATION
# ---------------------------------------------------------

def classify_ai_relevance_v2(row):

    title = str(row["title"]).lower().strip()
    description = clean_ai_description(row["description"])

    # =====================================================
    # A. STRONG AI TITLE
    # =====================================================

    if matches_any(title, AI_CORE_TITLE_PATTERNS):

        return "AI Required / Responsibility"

    # AI specialization attached to an actual technical /
    # product / research role

    if matches_any(title, AI_TITLE_SPECIALIZATION_PATTERNS):

        return "AI Required / Responsibility"

    # =====================================================
    # B. AI RESPONSIBILITY IN DESCRIPTION
    # =====================================================

    if matches_any(
        description,
        AI_RESPONSIBILITY_PATTERNS
    ):

        return "AI Required / Responsibility"

    # =====================================================
    # C. EXPLICITLY PREFERRED
    # =====================================================

    if matches_any(
        description,
        AI_PREFERRED_PATTERNS
    ):

        return "AI Preferred"

    # =====================================================
    # D. MULTIPLE AI TECHNOLOGIES
    # =====================================================

    ai_tech_count = count_ai_technologies(description)

    # Multiple distinct AI technologies in a non-AI title
    # indicate that AI is probably relevant to the actual work.

    if ai_tech_count >= 2:

        return "AI Required / Responsibility"

    # =====================================================
    # E. SINGLE MEANINGFUL AI TECHNOLOGY
    # =====================================================

    if ai_tech_count == 1:

        return "AI Mention Only"

    # =====================================================
    # F. GENERAL AI USAGE
    # =====================================================

    if matches_any(
        description,
        AI_USAGE_PATTERNS
    ):

        return "AI Mention Only"

    # =====================================================
    # G. NOTHING MEANINGFUL
    # =====================================================

    return "No AI Mention"


all_jobs["ai_relevance"] = all_jobs.apply(
    classify_ai_relevance_v2,
    axis=1
)

In [186]:
all_jobs["ai_relevance"].value_counts()

ai_relevance
No AI Mention                   982
AI Mention Only                 557
AI Required / Responsibility    158
AI Preferred                     54
Name: count, dtype: int64

In [187]:
all_jobs[
    all_jobs["ai_relevance"] == "AI Required / Responsibility"
][[
    "company",
    "title",
    "ai_keywords_found"
]].head(100)

,company,title,ai_keywords_found
70,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
71,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
72,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
73,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
74,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
...,...,...,...
1041,Stripe,Strategy & Transformation Advisor (Growth & Mo...,[]
1072,Stripe,Treasury Finance AI and Quantitative Analytics...,"[artificial intelligence, generative ai, deep ..."
1105,Twilio,Machine Learning Engineer,"[artificial intelligence, machine learning, ll..."
1106,Twilio,Machine Learning Engineer,"[artificial intelligence, machine learning, ll..."


In [188]:
all_jobs[
    all_jobs["ai_relevance"].isin(
        ["AI Preferred", "AI Mention Only"]
    )
][[
    "company",
    "title",
    "ai_relevance",
    "ai_keywords_found"
]].head(100)

,company,title,ai_relevance,ai_keywords_found
97,GitLab,Business Development Representative - MENA,AI Mention Only,[]
112,GitLab,"Director of Engineering, Organizations & Cells",AI Mention Only,[]
114,GitLab,"Director of Recruiting, Engineering & IT, Bang...",AI Mention Only,"[artificial intelligence, machine learning]"
115,GitLab,"Director, Strategic Partnerships",AI Mention Only,[]
126,GitLab,"Engineering Manager, Dedicated Integrations",AI Mention Only,[]
...,...,...,...,...
734,Stripe,Internal Audit Data Analytics Lead,AI Mention Only,[machine learning]
751,Stripe,"Manager, Global Sanctions",AI Preferred,[]
752,Stripe,"Manager, Global Sanctions",AI Preferred,[]
754,Stripe,"Manager, Payments Performance Strategist (SEA ...",AI Mention Only,[]


In [189]:
import re

# =========================================================
# FINAL AI CLASSIFIER - CONSERVATIVE VERSION
# =========================================================

# ---------------------------------------------------------
# AI TERMS THAT ARE SAFE TO USE
# ---------------------------------------------------------

AI_TERMS = [
    r"\bartificial intelligence\b",
    r"\bmachine learning\b",
    r"\bdeep learning\b",
    r"\blarge language models?\b",
    r"\bgenerative ai\b",
    r"\bgenai\b",
    r"\bai agents?\b",
    r"\bagentic ai\b",
    r"\bagentic\b",
    r"\bprompt engineering\b",
    r"\bfoundation models?\b",
    r"\bnatural language processing\b",
    r"\bnlp\b",
    r"\bcomputer vision\b",
    r"\bretrieval augmented generation\b",
    r"\brag pipelines?\b",
    r"\bembeddings?\b",
]

# ---------------------------------------------------------
# AI TITLE SIGNALS
# ---------------------------------------------------------

AI_TITLE_PATTERNS = [

    # Direct AI roles
    r"\bai engineer\b",
    r"\bai scientist\b",
    r"\bai researcher\b",
    r"\bai architect\b",
    r"\bai specialist\b",

    # ML roles
    r"\bmachine learning engineer\b",
    r"\bmachine learning scientist\b",
    r"\bmachine learning researcher\b",
    r"\bml engineer\b",
    r"\bml scientist\b",

    # AI engineering
    r"\bai engineering\b",
    r"\bai platform\b",
    r"\bai infrastructure\b",
    r"\bai systems?\b",

    # Agentic roles
    r"\bagentic platform\b",
    r"\bagentic engineering\b",
    r"\bagentic sdlc\b",
    r"\bai agents?\b",

    # AI product / strategy roles
    r"\bai product\b",
    r"\bai products\b",
    r"\bai strategy\b",
    r"\bai transformation\b",

    # AI-specific specialist roles
    r"\bai applied scientist\b",
    r"\bdata and ai specialist\b",
    r"\bai & platforms\b",

    # AI-specific titles
    r"\bai custom models?\b",
    r"\bai software factory\b",
]


# ---------------------------------------------------------
# AI SPECIALIZATION IN TITLE
# ---------------------------------------------------------

AI_SPECIALIZATION_PATTERNS = [

    r"\b(engineer|developer|architect|scientist|researcher|"
    r"designer|product manager|product lead|specialist)\b"
    r".*\b(ai|machine learning|ml|genai|generative ai|agentic)\b",

    r"\b(ai|machine learning|ml|genai|generative ai|agentic)\b"
    r".*\b(engineer|developer|architect|scientist|researcher|"
    r"designer|product manager|product lead|specialist)\b",
]


# ---------------------------------------------------------
# STRONG RESPONSIBILITY LANGUAGE
# ---------------------------------------------------------

AI_RESPONSIBILITY_PATTERNS = [

    r"\bresponsible for\b.{0,120}"
    r"\b(ai|machine learning|ml|generative ai|genai|"
    r"large language models?|ai agents?)\b",

    r"\bbuild\b.{0,100}"
    r"\b(ai systems?|ai products?|machine learning models?|"
    r"ml models?|llm applications?|generative ai|ai agents?)\b",

    r"\bdevelop\b.{0,100}"
    r"\b(ai systems?|ai products?|machine learning models?|"
    r"ml models?|llm applications?|generative ai|ai agents?)\b",

    r"\bdesign\b.{0,100}"
    r"\b(ai systems?|ai products?|machine learning models?|"
    r"ml models?|llm applications?|generative ai|ai agents?)\b",

    r"\bwork on\b.{0,100}"
    r"\b(ai systems?|ai products?|machine learning|ml|"
    r"llm applications?|generative ai|ai agents?)\b",

    r"\bworking on\b.{0,100}"
    r"\b(ai systems?|ai products?|machine learning|ml|"
    r"llm applications?|generative ai|ai agents?)\b",

    r"\bdeveloping\b.{0,100}"
    r"\b(ai systems?|ai products?|machine learning|ml|"
    r"llm applications?|generative ai|ai agents?)\b",
]


# ---------------------------------------------------------
# PREFERRED SIGNALS
# ---------------------------------------------------------

AI_PREFERRED_PATTERNS = [

    r"\b(ai|artificial intelligence|machine learning|"
    r"generative ai|genai)\b.{0,100}"
    r"\b(preferred|nice to have|nice-to-have|desirable|"
    r"a plus|plus)\b",

    r"\b(preferred|nice to have|nice-to-have|desirable|"
    r"a plus|plus)\b.{0,100}"
    r"\b(ai|artificial intelligence|machine learning|"
    r"generative ai|genai)\b",
]


# ---------------------------------------------------------
# REMOVE CORPORATE AI BOILERPLATE
# ---------------------------------------------------------

def remove_ai_boilerplate(text):

    text = str(text).lower()

    boilerplate = [

        r"we use artificial intelligence.*?(?:\.|$)",
        r"we use ai.*?(?:\.|$)",
        r"leverages ai to enhance our hiring.*?(?:\.|$)",
        r"ai.*?hiring process.*?(?:\.|$)",
        r"ai.*?recruitment process.*?(?:\.|$)",
        r"privacy and ai guidelines.*?(?:\.|$)",
        r"ai policy.*?(?:\.|$)",
        r"ai guidelines.*?(?:\.|$)",
    ]

    for pattern in boilerplate:

        text = re.sub(
            pattern,
            " ",
            text,
            flags=re.IGNORECASE
        )

    return text


# ---------------------------------------------------------
# REMOVE LLM LEGAL-DEGREE FALSE POSITIVES
# ---------------------------------------------------------

def remove_llm_false_positive(text):

    text = str(text)

    patterns = [

        r"\bllm in tax\b",
        r"\bllm in law\b",
        r"\bllm in corporate law\b",
        r"\bllm in international law\b",
        r"\bmaster of laws\b",
        r"\bmaster's degree in law\b",
    ]

    for pattern in patterns:

        text = re.sub(
            pattern,
            " ",
            text,
            flags=re.IGNORECASE
        )

    return text


# ---------------------------------------------------------
# HELPER
# ---------------------------------------------------------

def matches(text, patterns):

    return any(
        re.search(
            pattern,
            text,
            re.IGNORECASE
        )
        for pattern in patterns
    )


# ---------------------------------------------------------
# FINAL CLASSIFIER
# ---------------------------------------------------------

def classify_ai_final(row):

    title = str(row["title"]).lower().strip()

    description = remove_ai_boilerplate(
        row["description"]
    )

    description = remove_llm_false_positive(
        description
    )

    # =====================================================
    # 1. CLEAR AI TITLE
    # =====================================================

    if matches(title, AI_TITLE_PATTERNS):

        return "AI Required / Responsibility"


    # =====================================================
    # 2. AI SPECIALIZATION IN TECHNICAL / PRODUCT TITLE
    # =====================================================

    if matches(
        title,
        AI_SPECIALIZATION_PATTERNS
    ):

        return "AI Required / Responsibility"


    # =====================================================
    # 3. EXPLICIT AI RESPONSIBILITY
    # =====================================================

    if matches(
        description,
        AI_RESPONSIBILITY_PATTERNS
    ):

        return "AI Required / Responsibility"


    # =====================================================
    # 4. AI PREFERRED
    # =====================================================

    if matches(
        description,
        AI_PREFERRED_PATTERNS
    ):

        return "AI Preferred"


    # =====================================================
    # 5. MEANINGFUL AI MENTION
    # =====================================================

    meaningful_ai = matches(
        description,
        AI_TERMS
    )

    if meaningful_ai:

        return "AI Mention Only"


    # =====================================================
    # 6. NO AI
    # =====================================================

    return "No AI Mention"


all_jobs["ai_relevance"] = all_jobs.apply(
    classify_ai_final,
    axis=1
)

In [190]:
all_jobs["ai_relevance"].value_counts()

ai_relevance
No AI Mention                   1368
AI Mention Only                  205
AI Required / Responsibility     120
AI Preferred                      58
Name: count, dtype: int64

In [191]:
all_jobs[
    all_jobs["ai_relevance"] == "AI Required / Responsibility"
][[
    "company",
    "title",
    "ai_keywords_found"
]].head(100)

,company,title,ai_keywords_found
70,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
71,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
72,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
73,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
74,Careers at Tide,"Senior Staff Software Engineer, Agentic Platform",[llm]
...,...,...,...
1342,Datadog,"Director, Product Management - AI Observability","[machine learning, llm]"
1349,Datadog,"Distinguished Architect, AI",[llm]
1355,Datadog,"Engineering Manager I, Threat Detection",[]
1407,Datadog,Group Product Manager - Bring Your Own Cloud,[llm]


In [192]:
all_jobs[
    all_jobs["ai_relevance"] == "AI Preferred"
][[
    "company",
    "title",
    "ai_keywords_found"
]]

,company,title,ai_keywords_found
195,GitLab,"Senior Backend Engineer, Platform Readiness",[llm]
201,GitLab,"Senior Backend Engineer (Ruby on Rails), Plan:...",[artificial intelligence]
202,GitLab,"Senior Backend Engineer (Ruby), Plan: Portfoli...",[artificial intelligence]
218,GitLab,"Senior Engineering Manager, Non-Linear Product...",[]
219,GitLab,"Senior FP&A Analyst, Cloud Hosting",[]
242,GitLab,"Senior Program Manager, Enterprise Technology ...",[]
243,GitLab,"Senior Program Manager, Enterprise Technology ...",[]
279,GitLab,"Staff Backend Engineer (Go), Events Platform",[]
298,GitLab,"Staff Systems Engineer, IT",[]
540,Stripe,"Administrative Business Partner, Office of the...",[]


In [193]:
ai_required_pct = (
    (all_jobs["ai_relevance"] == "AI Required / Responsibility").sum()
    / len(all_jobs)
    * 100
)

ai_required_pct

np.float64(6.853226727584237)

In [194]:
all_jobs.dtypes

job_id                                    int64
title                                       str
company                                     str
location                                    str
url                                         str
published                   datetime64[us, UTC]
updated                     datetime64[us, UTC]
description                                 str
location_clean                              str
is_remote                                  bool
published_year                            int32
description_length                        int64
description_word_count                    int64
title_length                              int64
has_ai_in_title                            bool
has_ai_in_description                      bool
ai_keywords_found                        object
ai_keyword_count                          int64
ai_categories_found                      object
ai_category_count                         int64
ai_sentences                            

In [195]:
all_jobs.select_dtypes(include="number").describe().T

,count,mean,std,min,25%,50%,75%,max
job_id,1751.0,2.129194e+09,3.448928e+09,1497543.0,7955968.0,8096642.0,5.831267e+09,8.782040e+09
published_year,1751.0,2.025844e+03,5.361897e-01,2019.0,2026.0,2026.0,2.026000e+03,2.026000e+03
description_length,1751.0,5.906551e+03,1.950496e+03,2227.0,4400.0,5579.0,7.400500e+03,1.479100e+04
description_word_count,1751.0,8.523969e+02,2.838620e+02,268.0,635.0,801.0,1.067000e+03,1.964000e+03
title_length,1751.0,3.725186e+01,1.206122e+01,10.0,28.0,36.0,4.500000e+01,8.900000e+01
ai_keyword_count,1751.0,4.151913e-01,9.595106e-01,0.0,0.0,0.0,0.000000e+00,1.000000e+01
ai_category_count,1751.0,3.415191e-01,7.693305e-01,0.0,0.0,0.0,0.000000e+00,9.000000e+00
ai_sentence_count,1751.0,3.033124e+00,3.292596e+00,0.0,0.0,2.0,5.000000e+00,3.000000e+01


In [196]:
# Check important categorical columns
for col in ["company", "seniority", "role_type", "country", "location_type", "ai_relevance"]:
    print(f"\n--- {col} ---")
    print(all_jobs[col].value_counts(dropna=False))


--- company ---
company
Stripe             612
Datadog            444
GitLab             231
Figma              157
Twilio             144
Careers at Tide     84
Webflow             29
Customer.io         28
Karbon              15
BigID                7
Name: count, dtype: int64

--- seniority ---
seniority
Mid             770
Manager         516
Senior          193
Lead            174
Director         71
Entry/Junior     18
VP/Executive      9
Name: count, dtype: int64

--- role_type ---
role_type
Sales / Business Development         435
Engineering / Technology             331
Other                                219
Operations                           104
Product                              102
Professional Services / Solutions     92
Customer Support / Success            88
Marketing                             74
Security                              72
Finance / Accounting                  68
HR / Recruiting                       49
AI / Data Science                     47
Leg

In [197]:
print("Published date range:")
print(all_jobs["published"].min())
print(all_jobs["published"].max())

print("\nUpdated date range:")
print(all_jobs["updated"].min())
print(all_jobs["updated"].max())

print("\nPublished dates after updated dates:")
print((all_jobs["published"] > all_jobs["updated"]).sum())

Published date range:
2019-04-17 18:46:26+00:00
2026-09-04 02:24:22+00:00

Updated date range:
2025-07-14 21:29:03+00:00
2026-09-04 02:24:22+00:00

Published dates after updated dates:
0


In [198]:
numeric_cols = [
    "description_length",
    "description_word_count",
    "title_length",
    "ai_keyword_count",
    "ai_category_count",
    "ai_sentence_count"
]

for col in numeric_cols:
    print(f"\n--- {col} ---")
    print("Negative:", (all_jobs[col] < 0).sum())
    print("Zero:", (all_jobs[col] == 0).sum())
    print("Missing:", all_jobs[col].isna().sum())


--- description_length ---
Negative: 0
Zero: 0
Missing: 0

--- description_word_count ---
Negative: 0
Zero: 0
Missing: 0

--- title_length ---
Negative: 0
Zero: 0
Missing: 0

--- ai_keyword_count ---
Negative: 0
Zero: 1340
Missing: 0

--- ai_category_count ---
Negative: 0
Zero: 1340
Missing: 0

--- ai_sentence_count ---
Negative: 0
Zero: 486
Missing: 0


In [199]:
pd.crosstab(
    all_jobs["ai_relevance"],
    all_jobs["has_ai_in_title"]
)

has_ai_in_title,False,True
ai_relevance,,
AI Mention Only,196,9
AI Preferred,49,9
AI Required / Responsibility,40,80
No AI Mention,1355,13


In [200]:
pd.crosstab(
    all_jobs["ai_relevance"],
    all_jobs["has_ai_in_description"]
)

has_ai_in_description,False,True
ai_relevance,,
AI Mention Only,76,129
AI Preferred,36,22
AI Required / Responsibility,23,97
No AI Mention,1205,163


In [201]:
print("Rows:", len(all_jobs))
print("Unique job IDs:", all_jobs["job_id"].nunique())
print("Duplicate job IDs:", all_jobs["job_id"].duplicated().sum())

Rows: 1751
Unique job IDs: 1751
Duplicate job IDs: 0


In [202]:
final_jobs = all_jobs[
    [
        "job_id",
        "title",
        "company",
        "location",
        "location_clean",
        "country",
        "location_type",
        "is_remote",
        "published",
        "updated",
        "published_year",
        "seniority",
        "role_type",
        "ai_relevance",
        "has_ai_in_title",
        "has_ai_in_description",
        "ai_keyword_count",
        "ai_category_count",
        "ai_sentence_count",
        "description_length",
        "description_word_count",
        "title_length",
        "url",
        "description"
    ]
].copy()

print("Final shape:", final_jobs.shape)
print("\nColumns:")
print(final_jobs.columns.tolist())

Final shape: (1751, 24)

Columns:
['job_id', 'title', 'company', 'location', 'location_clean', 'country', 'location_type', 'is_remote', 'published', 'updated', 'published_year', 'seniority', 'role_type', 'ai_relevance', 'has_ai_in_title', 'has_ai_in_description', 'ai_keyword_count', 'ai_category_count', 'ai_sentence_count', 'description_length', 'description_word_count', 'title_length', 'url', 'description']


In [203]:
from datetime import datetime, timezone

final_jobs = final_jobs.rename(columns={
    "published": "published_at",
    "updated": "updated_at"
})

final_jobs["scraped_at"] = datetime.now(timezone.utc)

print("Final shape:", final_jobs.shape)
print(final_jobs.columns.tolist())
print("\nScraped at:", final_jobs["scraped_at"].iloc[0])

Final shape: (1751, 25)
['job_id', 'title', 'company', 'location', 'location_clean', 'country', 'location_type', 'is_remote', 'published_at', 'updated_at', 'published_year', 'seniority', 'role_type', 'ai_relevance', 'has_ai_in_title', 'has_ai_in_description', 'ai_keyword_count', 'ai_category_count', 'ai_sentence_count', 'description_length', 'description_word_count', 'title_length', 'url', 'description', 'scraped_at']

Scraped at: 2026-09-04 02:41:33.314880+00:00


In [204]:
final_jobs.dtypes

job_id                                  int64
title                                     str
company                                   str
location                                  str
location_clean                            str
country                                   str
location_type                             str
is_remote                                bool
published_at              datetime64[us, UTC]
updated_at                datetime64[us, UTC]
published_year                          int32
seniority                                 str
role_type                                 str
ai_relevance                              str
has_ai_in_title                          bool
has_ai_in_description                    bool
ai_keyword_count                        int64
ai_category_count                       int64
ai_sentence_count                       int64
description_length                      int64
description_word_count                  int64
title_length                      

In [205]:
final_jobs.to_csv("final_jobs.csv", index=False)

print("Saved successfully.")

Saved successfully.


In [206]:
import os

print("File exists:", os.path.exists("final_jobs.csv"))
print("File size (MB):", round(os.path.getsize("final_jobs.csv") / (1024 * 1024), 2))

File exists: True
File size (MB): 10.54


In [207]:
mysql_jobs = final_jobs.copy()

# Convert timezone-aware timestamps to MySQL DATETIME format
for col in ["published_at", "updated_at", "scraped_at"]:
    mysql_jobs[col] = (
        mysql_jobs[col]
        .dt.tz_convert(None)
        .dt.strftime("%Y-%m-%d %H:%M:%S")
    )

# Convert booleans to MySQL-friendly 0/1
for col in ["is_remote", "has_ai_in_title", "has_ai_in_description"]:
    mysql_jobs[col] = mysql_jobs[col].astype(int)

# Save MySQL-compatible copy
mysql_jobs.to_csv("final_jobs_mysql.csv", index=False)

print("Saved successfully.")
print("Shape:", mysql_jobs.shape)

Saved successfully.
Shape: (1751, 25)


In [208]:
print(mysql_jobs[
    [
        "published_at",
        "updated_at",
        "is_remote",
        "has_ai_in_title",
        "has_ai_in_description",
        "scraped_at"
    ]
].head())

          published_at           updated_at  is_remote  has_ai_in_title  \
0  2026-03-27 13:45:37  2026-08-28 10:01:23          0                0   
1  2025-10-31 14:40:09  2026-08-28 10:01:23          0                0   
2  2026-04-01 12:43:11  2026-08-28 10:01:23          0                0   
3  2026-06-12 09:01:36  2026-08-28 10:01:23          0                0   
4  2026-07-30 08:09:40  2026-08-28 10:01:23          0                0   

   has_ai_in_description           scraped_at  
0                      0  2026-09-04 02:41:33  
1                      0  2026-09-04 02:41:33  
2                      0  2026-09-04 02:41:33  
3                      0  2026-09-04 02:41:33  
4                      0  2026-09-04 02:41:33  


In [209]:
print("Rows:", len(mysql_jobs))
print("Columns:", len(mysql_jobs.columns))
print("Unique job IDs:", mysql_jobs["job_id"].nunique())

Rows: 1751
Columns: 25
Unique job IDs: 1751


In [210]:
print("all_jobs:", all_jobs.shape)
print("final_jobs:", final_jobs.shape)
print("mysql_jobs:", mysql_jobs.shape)

all_jobs: (1751, 29)
final_jobs: (1751, 25)
mysql_jobs: (1751, 25)


In [211]:
print(
    final_jobs["company"].value_counts()
)

company
Stripe             612
Datadog            444
GitLab             231
Figma              157
Twilio             144
Careers at Tide     84
Webflow             29
Customer.io         28
Karbon              15
BigID                7
Name: count, dtype: int64


In [212]:
print(
    mysql_jobs["company"].value_counts()
)

company
Stripe             612
Datadog            444
GitLab             231
Figma              157
Twilio             144
Careers at Tide     84
Webflow             29
Customer.io         28
Karbon              15
BigID                7
Name: count, dtype: int64


In [213]:
print("Rows:", len(final_jobs))
print("Unique IDs:", final_jobs["job_id"].nunique())
print("Duplicate IDs:", final_jobs["job_id"].duplicated().sum())
print("Missing values:", final_jobs.isna().sum().sum())

Rows: 1751
Unique IDs: 1751
Duplicate IDs: 0
Missing values: 0


In [214]:
print(final_jobs["ai_relevance"].value_counts())

ai_relevance
No AI Mention                   1368
AI Mention Only                  205
AI Required / Responsibility     120
AI Preferred                      58
Name: count, dtype: int64


In [215]:
mysql_jobs = final_jobs.copy()

for col in ["published_at", "updated_at", "scraped_at"]:
    mysql_jobs[col] = (
        mysql_jobs[col]
        .dt.tz_convert(None)
        .dt.strftime("%Y-%m-%d %H:%M:%S")
    )

for col in ["is_remote", "has_ai_in_title", "has_ai_in_description"]:
    mysql_jobs[col] = mysql_jobs[col].astype(int)

mysql_jobs.to_csv("final_jobs_mysql.csv", index=False)

print("Rows:", len(mysql_jobs))
print("Columns:", len(mysql_jobs.columns))
print("Unique IDs:", mysql_jobs["job_id"].nunique())

Rows: 1751
Columns: 25
Unique IDs: 1751


In [216]:
import pandas as pd

sql_file = "final_jobs_import.sql"

columns = [
    "job_id",
    "title",
    "company",
    "location",
    "location_clean",
    "country",
    "location_type",
    "is_remote",
    "published_at",
    "updated_at",
    "published_year",
    "seniority",
    "role_type",
    "ai_relevance",
    "has_ai_in_title",
    "has_ai_in_description",
    "ai_keyword_count",
    "ai_category_count",
    "ai_sentence_count",
    "description_length",
    "description_word_count",
    "title_length",
    "url",
    "description",
    "scraped_at"
]

def sql_value(value):
    if pd.isna(value):
        return "NULL"

    if isinstance(value, str):
        # Escape backslashes and single quotes for MySQL
        value = value.replace("\\", "\\\\")
        value = value.replace("'", "''")
        return f"'{value}'"

    return str(value)


with open(sql_file, "w", encoding="utf-8") as f:

    f.write("USE my_jobs_db;\n\n")

    f.write("TRUNCATE TABLE job_postings;\n\n")

    batch_size = 50

    for start in range(0, len(mysql_jobs), batch_size):

        batch = mysql_jobs.iloc[start:start + batch_size]

        f.write(
            "INSERT INTO job_postings "
            f"({', '.join(columns)}) VALUES\n"
        )

        rows = []

        for _, row in batch.iterrows():
            values = [
                sql_value(row[col])
                for col in columns
            ]

            rows.append("(" + ", ".join(values) + ")")

        f.write(",\n".join(rows))
        f.write(";\n\n")

print("SQL file created successfully.")
print("Rows:", len(mysql_jobs))
print("File:", sql_file)

SQL file created successfully.
Rows: 1751
File: final_jobs_import.sql


In [217]:
import pandas as pd

sql_file = "final_jobs_import_utf8.sql"

columns = [
    "job_id",
    "title",
    "company",
    "location",
    "location_clean",
    "country",
    "location_type",
    "is_remote",
    "published_at",
    "updated_at",
    "published_year",
    "seniority",
    "role_type",
    "ai_relevance",
    "has_ai_in_title",
    "has_ai_in_description",
    "ai_keyword_count",
    "ai_category_count",
    "ai_sentence_count",
    "description_length",
    "description_word_count",
    "title_length",
    "url",
    "description",
    "scraped_at"
]

def sql_value(value):
    if pd.isna(value):
        return "NULL"

    if isinstance(value, str):
        # Encode Unicode text as UTF-8 hex.
        # This safely handles emojis and other 4-byte characters.
        hex_value = value.encode("utf-8").hex()
        return f"CONVERT(X'{hex_value}' USING utf8mb4)"

    return str(value)


with open(sql_file, "w", encoding="ascii") as f:

    f.write("USE my_jobs_db;\n\n")
    f.write("TRUNCATE TABLE job_postings;\n\n")

    batch_size = 50

    for start in range(0, len(mysql_jobs), batch_size):

        batch = mysql_jobs.iloc[start:start + batch_size]

        f.write(
            "INSERT INTO job_postings "
            f"({', '.join(columns)}) VALUES\n"
        )

        rows = []

        for _, row in batch.iterrows():

            values = [
                sql_value(row[col])
                for col in columns
            ]

            rows.append("(" + ", ".join(values) + ")")

        f.write(",\n".join(rows))
        f.write(";\n\n")

print("Unicode-safe SQL file created.")
print("Rows:", len(mysql_jobs))
print("Columns:", len(mysql_jobs.columns))
print("File:", sql_file)

Unicode-safe SQL file created.
Rows: 1751
Columns: 25
File: final_jobs_import_utf8.sql
